### Middleware
Middeware provides a way to more tightly control what happens inside the agent Middleware is useful for the
following:

 - Tracking agent behavior with logging, analytics, and debugging_
 - Transforming prompts, tool selection, and output formatting.
 - Adding retries, fallbacks, and early termination logic.
 - Applying rate limits, guardrails, and PII detection.

In [ ]:
import os 
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY") 

### Summarization MiddleWare
Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

• Long-running conversations that exceed context windows.

• Multi-turn dialogues with extensive history.

• Applications where preserving full conversation context matters.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage, SystemMessage

# Message Based Summarization
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware (
            model="groq:qwen/qwen3-32b",
            trigger=("messages",10),
            keep=("messages",4)
        )
    ]
)

In [3]:
### Run with a thread id
config={"configurable":{"thread_id":"test-1"}}

In [4]:
# ALternative test data
questions = [
    "What is 2+2",
    "What is 10*8",
    "What is 100/4",
    "What is 214-11",
    "What is 20*2",
    "What is 12*21",
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2', additional_kwargs={}, response_metadata={}, id='efc2f0ed-0786-427a-b69f-2caa37f1ad6a'), AIMessage(content='<think>\nOkay, so I need to figure out what 2+2 is. Hmm, let\'s start by recalling basic addition. Addition is combining two numbers to find their total. So 2 plus 2 would be adding two and another two together. Let me visualize this. If I have two apples and then get two more apples, how many apples do I have in total? That would be four apples. \n\nWait, maybe I should break it down step by step. The number 2 represents a quantity, and when we add another 2, we\'re increasing that quantity by the same amount. So starting from 2, if I count two more: 2, 3, 4. Yeah, that makes sense. Each increment is one step. So two steps from 2 would be 4.\n\nAnother way to think about it is using physical objects. If I have two blocks and place two more blocks next to them, the total number of blocks is four. I can even draw it out. T

### Token Size

In [9]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage 
from langchain_core.tools import tool

@tool
def search_hotels (city:str) -> str:
    """Search hotels - returns long responses to use more tokens.""" 
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""

agent = create_agent(
    model="groq:qwen/qwen3-32b", 
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("tokens", 550),
            keep=("tokens",200),
        )
    ]
)

config = {"configurable":{"thread_id":"test-1"}}

# Token Counter
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4 # 4 chars = 1 token


In [10]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response =  agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config = config
    )

    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])}messages")
    print(f"{(response['messages'])}")

Paris: ~155 tokens, 4messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='c9a7fe55-39fb-463b-a211-78f653dbd7aa'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants to find hotels in Paris. Let me check the available tools. There\'s a function called search_hotels that takes a city parameter. The description says it returns long responses to use more tokens. Hmm, maybe the response is detailed, which is good for the user. I need to call this function with the city set to Paris. Let me make sure the parameters are correct. The required field is city, and it\'s a string. So the arguments should be {"city": "Paris"}. I\'ll structure the tool call accordingly.\n', 'tool_calls': [{'id': 'bny3g8ghk', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 138, 'prompt_tokens': 155, 'total_tokens': 293, 'completion_ti

### Fraction

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~66 tokens (0.0516%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='1c11142e-b329-4978-b1c3-c3e99d1466bc'), AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for hotels in Paris. Let me check the available tools. There's a function called search_hotels that takes a city parameter. Since the user specified Paris, I need to call that function with the city set to Paris. I'll make sure the arguments are correctly formatted as JSON within the tool_call tags.\n", 'tool_calls': [{'id': 'sqw7db1e4', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 147, 'total_tokens': 240, 'completion_time': 0.136444893, 'completion_tokens_details': {'reasoning_tokens': 68}, 'prompt_time': 0.006771401, 'prompt_tokens_details': None, 'queue_time': 0.055198139, 'total_time': 0.143216294}, 'model_n